# 02 · Microsoft Agent Framework로 같은 동작 만들기

## 학습 목표
- 모델 클라이언트·지침·도구·대화 세션으로 에이전트를 조립합니다.
- 01의 수동 루프를 Framework가 수행하는 것을 관찰합니다.

## 개념
**에이전트 = LLM + 지침 + 도구 + 실행 제어 + 상태**. 도구를 많이 넣는 것 자체가 좋은 설계는 아닙니다. 이번 실습은 읽기 전용 도구만 가진 단일 에이전트입니다.

검증 버전의 `OpenAIChatClient`는 **Responses API**를 사용합니다. 예전 문서의 `OpenAIResponsesClient`는 현재 이름이 아닙니다. `OpenAIChatCompletionClient`는 별도의 Chat Completions 클라이언트입니다.

In [ ]:
from pathlib import Path
import sys
import json
from dotenv import load_dotenv

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
load_dotenv(ROOT / ".env")
from hanbit_assistant.config import Settings
from hanbit_assistant.clients import create_openai_client
settings = Settings.from_env()
# 비밀 키, 환경변수 전체, HTTP 헤더를 출력하지 않습니다.
print({"model": settings.model, "my_index": settings.search_index})


In [ ]:
from agent_framework import Agent
from agent_framework.openai import OpenAIChatClient
from hanbit_assistant.tools import get_weather as weather_api

async def current_weather(city: str) -> str:
    """서울, 도쿄, 런던, 뉴욕의 현재 날씨를 조회한다. 실시간 날씨 질문에 사용한다."""
    print("실제로 실행된 함수:", "current_weather", city)
    return json.dumps(await weather_api(city), ensure_ascii=False)

async with create_openai_client(settings) as api:
    model_client = OpenAIChatClient(model=settings.model, async_client=api,
        function_invocation_configuration={"max_iterations": 4, "max_function_calls": 3,
                                           "allow_concurrent_invocation": False})
    agent = Agent(client=model_client, name="weather-assistant",
        instructions="한국어 출장 도우미. 현재 날씨는 반드시 도구로 확인하고 기준 시각과 출처를 표시하라. 실패 시 추측하지 마라.",
        tools=[current_weather], default_options={"store": False, "max_tokens": 650})
    session = agent.create_session()
    answer = await agent.run("서울 현재 날씨를 알려줘.", session=session)
    print(answer.text)
    followup = await agent.run("방금 조회한 도시의 이름만 말해줘.", session=session)
    print("후속 답변:", followup.text)


## 관찰
별도의 `while` 함수 실행 루프를 쓰지 않았지만 실제 함수가 실행됩니다. 세션은 로컬 대화 기록을 유지하며, 서버의 `previous_response_id`를 사용하지 않습니다. 브라우저 사용자 간 세션을 전역 공유하면 안 됩니다.

## 연습
환율 함수를 두 번째 도구로 추가하려면 무엇이 필요할까요? 로컬 함수·입력 설명·에이전트 `tools` 목록을 확인하세요.

<details><summary>예시 해설</summary>

```python
async def exchange(amount: float, from_currency: str, to_currency: str) -> str:
    """최근 영업일 참고 환율로 금액을 환산한다."""
    return json.dumps(await convert_currency(amount, from_currency, to_currency), ensure_ascii=False)
# tools=[current_weather, exchange]
```

Framework는 함수 서명·설명에서 스키마를 만듭니다. 타입 검증뿐 아니라 금액 상한·지원 통화·타임아웃은 함수 내부에서도 검사합니다.
</details>

**다음 단계:** 회사 규정을 찾는 검색도 하나의 도구로 연결합니다.